# 15: The Crystal Gets a Past

State is enough for faithful continuation. History is enough to reconstruct how the morphology formed. They are related but not the same object.


In [1]:
from pathlib import Path
import json
import math
import random
import sys
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BOOK_FIG_DIR = ROOT / "static" / "images" / "books" / "digital-life"
FIG_DIR = ROOT / "notebooks" / "generated-figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "notebooks" / "_shared"))

from digital_crystal import (
    CrystalParams, CrystalState, neighbors, hex_distance, hex_capacity,
    axial_to_xy, logistic, local_exposure_angle, initial_state, clone_state,
    morphology_hash, frontier, attachment_probability, advance_one_step,
    run_crystal, cell_keyed_uniform,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180

NOTEBOOK_PROFILE = "quick"
RUN_CANONICAL = False

def load_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))

def audit(expected, observed, source):
    return {
        "source": source,
        "expected": expected,
        "observed": observed,
        "matches": {k: observed.get(k) == v for k, v in expected.items()},
    }

def plot_cells(cells, title, ax=None, color="tab:blue"):
    ax = ax or plt.gca()
    pts = np.array([axial_to_xy(c) for c in cells]) if cells else np.empty((0, 2))
    if len(pts):
        ax.scatter(pts[:, 0], pts[:, 1], marker="h", s=42, c=color, alpha=0.9)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    return ax


In [2]:
EXPERIMENT = {"quick_steps": 28, "checkpoint": 14, "max_radius": 22, "seed": 15, "canonical_profile": {"steps": 96, "checkpoint": 48, "max_radius": 56, "replicates": 30}}
EXPERIMENT


{'quick_steps': 28,
 'checkpoint': 14,
 'max_radius': 22,
 'seed': 15,
 'canonical_profile': {'steps': 96,
  'checkpoint': 48,
  'max_radius': 56,
  'replicates': 30}}

## Hidden implementation-state bug: unordered frontier consumes RNG differently


In [3]:
front = {(1, 0), (0, 1), (-1, 1), (2, -1)}
rng_values = [0.1, 0.2, 0.3, 0.4]
assignment_a = dict(zip(list(front), rng_values))
assignment_b = dict(zip(list(reversed(list(front))), rng_values))
sorted_assignment_a = dict(zip(sorted(front), rng_values))
sorted_assignment_b = dict(zip(sorted(front), rng_values))
{"unordered_can_reassign_values": assignment_a != assignment_b, "sorted_is_stable": sorted_assignment_a == sorted_assignment_b}


{'unordered_can_reassign_values': True, 'sorted_is_stable': True}

## Checkpoint -> exact future in quick mode


In [4]:
signal = np.linspace(0.2, 0.8, EXPERIMENT["quick_steps"])
state = initial_state(EXPERIMENT["seed"])
checkpoint = None
history_hashes = []
for i, value in enumerate(signal):
    if i == EXPERIMENT["checkpoint"]:
        checkpoint = clone_state(state)
    state, _ = advance_one_step(state, float(value), EXPERIMENT["max_radius"])
    history_hashes.append(morphology_hash(state))
reference = state
restored = clone_state(checkpoint)
for value in signal[EXPERIMENT["checkpoint"]:]:
    restored, _ = advance_one_step(restored, float(value), EXPERIMENT["max_radius"])
quick_restore = {
    "exact_morphology": restored.occupied == reference.occupied,
    "population_trajectory_equal": restored.population_by_step == reference.population_by_step[:EXPERIMENT["checkpoint"]+1] + reference.population_by_step[EXPERIMENT["checkpoint"]+1:],
    "symdiff": len(restored.occupied ^ reference.occupied),
}
quick_restore


{'exact_morphology': True, 'population_trajectory_equal': True, 'symdiff': 0}

## Event history -> exact recorded morphology past


In [5]:
events = []
state = initial_state(EXPERIMENT["seed"])
for value in signal:
    before = set(state.occupied)
    state, _ = advance_one_step(state, float(value), EXPERIMENT["max_radius"])
    events.append(sorted(state.occupied - before))
replay_cells = {(0, 0)}
replay_hashes = []
for additions in events:
    replay_cells |= set(map(tuple, additions))
    replay_hashes.append(morphology_hash(replay_cells))
{"event_replay_hashes_match": replay_hashes == history_hashes, "steps": len(replay_hashes)}


{'event_replay_hashes_match': True, 'steps': 28}

## Canonical artifact audit


In [6]:
ref = load_json("research/digital-life/ch15-reports/stage-01-reference.json")
restore = load_json("research/digital-life/ch15-reports/stage-02-exact-restore.json")
reps = load_json("research/digital-life/ch15-reports/stage-02b-restore-replicates.json")
omit = load_json("research/digital-life/ch15-reports/stage-03-state-omission.json")
replay = load_json("research/digital-life/ch15-reports/stage-04-history-replay.json")
observed = {
    "checkpoint_population": ref["checkpoint_population"],
    "final_population": ref["final_population"],
    "morphology_hash": ref["final_morphology_hash"],
    "process_hash": ref["final_process_hash"],
    "exact_final_morphology": restore["exact_morphology"],
    "exact_process_state": restore["exact_process_state"],
    "symmetric_difference": restore["symmetric_difference_cells"],
    "exact_replicates": f'{reps["exact_count"]}/{reps["replicates"]}',
    "history_replay_matches": replay["trajectory_hash_match_count"],
}
EXPECTED = {
    "checkpoint_population": 2702,
    "final_population": 9574,
    "morphology_hash": "bc8e6d8c9783431f1459bf17",
    "process_hash": "bd33c9aa0a510803be6ce6bf",
    "exact_final_morphology": True,
    "exact_process_state": True,
    "symmetric_difference": 0,
    "exact_replicates": "30/30",
    "history_replay_matches": 96,
}
audit(EXPECTED, observed, "canonical report artifact")


{'source': 'canonical report artifact',
 'expected': {'checkpoint_population': 2702,
  'final_population': 9574,
  'morphology_hash': 'bc8e6d8c9783431f1459bf17',
  'process_hash': 'bd33c9aa0a510803be6ce6bf',
  'exact_final_morphology': True,
  'exact_process_state': True,
  'symmetric_difference': 0,
  'exact_replicates': '30/30',
  'history_replay_matches': 96},
 'observed': {'checkpoint_population': 2702,
  'final_population': 9574,
  'morphology_hash': 'bc8e6d8c9783431f1459bf17',
  'process_hash': 'bd33c9aa0a510803be6ce6bf',
  'exact_final_morphology': True,
  'exact_process_state': True,
  'symmetric_difference': 0,
  'exact_replicates': '30/30',
  'history_replay_matches': 96},
 'matches': {'checkpoint_population': True,
  'final_population': True,
  'morphology_hash': True,
  'process_hash': True,
  'exact_final_morphology': True,
  'exact_process_state': True,
  'symmetric_difference': True,
  'exact_replicates': True,
  'history_replay_matches': True}}

In [7]:
variants = omit["variants"]
{k: {
    "final_population": v["final_population"],
    "symdiff": v["symmetric_difference_cells"],
    "normalized_difference": round(v["normalized_difference"], 6),
} for k, v in variants.items()}


{'full_state': {'final_population': 9574,
  'symdiff': 0,
  'normalized_difference': 0.0},
 'no_rng_state': {'final_population': 9550,
  'symdiff': 28,
  'normalized_difference': 0.002924},
 'wrong_signal_cursor_fixed_horizon': {'final_population': 9549,
  'symdiff': 27,
  'normalized_difference': 0.00282},
 'birth_times_only': {'final_population': 9574,
  'symdiff': 0,
  'normalized_difference': 0.0},
 'morphology_only': {'final_population': 9548,
  'symdiff': 30,
  'normalized_difference': 0.003133}}

## Bounded conclusion

Complete checkpoint state restored exact future. Event history replayed exact morphology past. Morphology alone and missing RNG/signal cursor diverged. The event log is not cognitive memory.

## Provenance

Book chapter: `content/books/digital-life/15-the-crystal-gets-a-past/index.md`

Research scripts: `ch15_digital_crystal_history.py`, `ch15_digital_crystal_history_v2.py`, `ch15_digital_crystal_history_v3.py`

Canonical artifacts: `research/digital-life/ch15-reports/`, `research/digital-life/ch15-digital-crystal-history.sqlite3`

Associated figures: `ch15-01-reference-and-checkpoint.png`, `ch15-02-exact-restore.png`, `ch15-03-state-omission.png`, `ch15-04-history-replay.png`, `ch15-06-*`

Execution mode: quick-recomputed + canonical-artifact-audit
